# Static Batching for LLM Inference

**Stage 1: Basic Optimization**

This notebook demonstrates static batching, which provides:
- **Higher throughput**: Process multiple requests simultaneously
- **Better GPU utilization**: Keep GPU busy with parallel work
- **Trade-off**: Increased latency per request vs higher total throughput

## What is Static Batching?

Static batching groups multiple inference requests together and processes them in parallel:
- **Without batching**: Process one request at a time (low GPU utilization)
- **With batching**: Process N requests together (higher GPU utilization)

### Key Concepts
1. **Padding**: All sequences in a batch must have the same length
2. **Throughput**: Total tokens/second across all requests
3. **Latency**: Time for a single request (increases with batch size)
4. **Efficiency**: Ratio of compute vs padding overhead

### Trade-offs
- **Larger batches**: Higher throughput, higher latency, more padding waste
- **Smaller batches**: Lower latency, lower throughput, less padding waste

**Reference**: See `LLM_INFERENCE_ROADMAP.md` - Stage 1, Basic Optimizations

In [ ]:
# Install required packages
!pip install torch transformers accelerate matplotlib pandas numpy tqdm -q

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple
from tqdm.auto import tqdm
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 1. Understanding Padding

Static batching requires padding sequences to the same length.

In [ ]:
# Load model and tokenizer
model_name = "gpt2"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
model.eval()

if device == "cpu":
    model = model.to(device)

print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

In [ ]:
# Demonstrate padding
def visualize_padding(prompts: List[str], tokenizer):
    """Visualize how padding works."""
    
    print("Padding Visualization")
    print("="*100)
    
    # Tokenize without padding
    token_lengths = []
    for i, prompt in enumerate(prompts):
        tokens = tokenizer(prompt, return_tensors="pt").input_ids[0]
        token_lengths.append(len(tokens))
        print(f"Prompt {i+1} ({len(tokens):2d} tokens): {prompt[:50]}..." if len(prompt) > 50 else f"Prompt {i+1} ({len(tokens):2d} tokens): {prompt}")
    
    max_length = max(token_lengths)
    
    print(f"\nMax length: {max_length} tokens")
    print(f"\nPadding overhead per prompt:")
    
    total_tokens_no_padding = sum(token_lengths)
    total_tokens_with_padding = max_length * len(prompts)
    padding_tokens = total_tokens_with_padding - total_tokens_no_padding
    
    for i, length in enumerate(token_lengths):
        padding = max_length - length
        efficiency = length / max_length * 100
        print(f"  Prompt {i+1}: {padding:2d} padding tokens ({100-efficiency:.1f}% waste, {efficiency:.1f}% efficient)")
    
    overall_efficiency = total_tokens_no_padding / total_tokens_with_padding * 100
    
    print(f"\nOverall batch efficiency: {overall_efficiency:.1f}%")
    print(f"Total padding overhead: {padding_tokens} tokens ({100-overall_efficiency:.1f}% of batch)")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 6))
    
    y_pos = np.arange(len(prompts))
    
    # Plot actual tokens
    ax.barh(y_pos, token_lengths, label='Actual tokens', color='lightblue')
    
    # Plot padding
    padding_lengths = [max_length - length for length in token_lengths]
    ax.barh(y_pos, padding_lengths, left=token_lengths, label='Padding', color='coral', alpha=0.6)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f'Prompt {i+1}' for i in range(len(prompts))])
    ax.set_xlabel('Tokens', fontsize=12)
    ax.set_title('Static Batching: Padding Overhead', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add efficiency annotations
    for i, (actual, padding) in enumerate(zip(token_lengths, padding_lengths)):
        efficiency = actual / max_length * 100
        ax.text(max_length + 1, i, f'{efficiency:.0f}% efficient', va='center')
    
    plt.tight_layout()
    plt.savefig('padding_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return overall_efficiency


# Test with varying length prompts
test_prompts = [
    "Hello",
    "The quick brown fox",
    "Artificial intelligence is transforming the world",
    "In the age of rapid technological advancement, we must consider the implications of AI",
]

efficiency = visualize_padding(test_prompts, tokenizer)

print("\nKey Insight: Padding efficiency depends on prompt length variance!")
print("  - Similar length prompts: High efficiency (low padding waste)")
print("  - Varying length prompts: Low efficiency (high padding waste)")

## 2. Implementing Static Batching

In [ ]:
def generate_sequential(model, tokenizer, prompts: List[str], max_new_tokens: int = 50):
    """Generate text for multiple prompts sequentially (no batching)."""
    outputs = []
    
    for prompt in prompts:
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        outputs.append(generated_text)
    
    return outputs


def generate_batched(model, tokenizer, prompts: List[str], max_new_tokens: int = 50):
    """Generate text for multiple prompts using static batching."""
    
    # Tokenize with padding
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,  # Pad to longest sequence in batch
        truncation=True,
    ).to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode all outputs
    outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    
    return outputs


# Test both approaches
test_prompts = [
    "The future of AI is",
    "Machine learning enables",
    "Deep neural networks can",
]

print("Testing sequential vs batched generation...\n")
print("="*80)

# Sequential
print("\nSequential generation (no batching):")
start = time.time()
seq_outputs = generate_sequential(model, tokenizer, test_prompts, max_new_tokens=30)
if device == "cuda":
    torch.cuda.synchronize()
seq_time = time.time() - start

for i, output in enumerate(seq_outputs):
    print(f"  {i+1}. {output}")
print(f"\nTime: {seq_time:.3f}s")

# Batched
print("\n" + "="*80)
print("\nBatched generation (static batching):")
start = time.time()
batch_outputs = generate_batched(model, tokenizer, test_prompts, max_new_tokens=30)
if device == "cuda":
    torch.cuda.synchronize()
batch_time = time.time() - start

for i, output in enumerate(batch_outputs):
    print(f"  {i+1}. {output}")
print(f"\nTime: {batch_time:.3f}s")

print("\n" + "="*80)
print(f"\nSpeedup: {seq_time / batch_time:.2f}x")
print(f"Throughput improvement: {(seq_time / batch_time - 1) * 100:.1f}%")

## 3. Throughput vs Latency Trade-off

In [ ]:
def benchmark_batch_size(
    model,
    tokenizer,
    prompts: List[str],
    batch_sizes: List[int],
    max_new_tokens: int = 50,
    num_runs: int = 3,
) -> pd.DataFrame:
    """Benchmark different batch sizes."""
    
    results = []
    
    for batch_size in batch_sizes:
        print(f"\nBenchmarking batch_size={batch_size}...")
        
        # Create batches from available prompts
        num_full_batches = len(prompts) // batch_size
        
        if num_full_batches == 0:
            print(f"  Skipping: not enough prompts for batch_size={batch_size}")
            continue
        
        times = []
        
        for run in range(num_runs):
            total_time = 0
            
            # Process all full batches
            for i in range(num_full_batches):
                batch_prompts = prompts[i * batch_size : (i + 1) * batch_size]
                
                start = time.time()
                _ = generate_batched(model, tokenizer, batch_prompts, max_new_tokens)
                if device == "cuda":
                    torch.cuda.synchronize()
                total_time += time.time() - start
            
            times.append(total_time)
        
        avg_time = np.mean(times)
        num_requests = num_full_batches * batch_size
        
        # Calculate metrics
        throughput = (num_requests * max_new_tokens) / avg_time  # tokens/sec
        latency_per_request = avg_time / num_requests  # seconds per request
        requests_per_sec = num_requests / avg_time
        
        results.append({
            'Batch Size': batch_size,
            'Total Time (s)': avg_time,
            'Latency/Request (s)': latency_per_request,
            'Throughput (tok/s)': throughput,
            'Requests/sec': requests_per_sec,
            'Num Requests': num_requests,
        })
        
        print(f"  Latency per request: {latency_per_request:.3f}s")
        print(f"  Throughput: {throughput:.1f} tokens/sec")
        print(f"  Requests/sec: {requests_per_sec:.2f}")
    
    return pd.DataFrame(results)


# Generate many test prompts
prompt_templates = [
    "The future of {} is",
    "In the world of {}, we see",
    "Recent advances in {} have shown",
    "The key to understanding {} is",
    "Researchers in {} have discovered",
]

topics = [
    "artificial intelligence", "machine learning", "deep learning",
    "natural language processing", "computer vision", "robotics",
    "quantum computing", "biotechnology", "renewable energy",
    "space exploration", "nanotechnology", "cybersecurity",
]

benchmark_prompts = [template.format(topic) for template in prompt_templates for topic in topics]
print(f"\nGenerated {len(benchmark_prompts)} test prompts")

# Benchmark different batch sizes
batch_sizes = [1, 2, 4, 8, 16]
print("\n" + "="*80)
print("Starting throughput vs latency benchmark...")
print("="*80)

df_results = benchmark_batch_size(
    model, tokenizer, benchmark_prompts,
    batch_sizes=batch_sizes,
    max_new_tokens=50,
    num_runs=3
)

In [ ]:
# Display results
print("\n" + "="*80)
print("Benchmark Results")
print("="*80)
print(df_results.to_string(index=False))

# Calculate relative improvements
baseline_throughput = df_results.iloc[0]['Throughput (tok/s)']
baseline_latency = df_results.iloc[0]['Latency/Request (s)']

print("\n" + "="*80)
print("Relative to batch_size=1:")
print("="*80)
for _, row in df_results.iterrows():
    throughput_gain = row['Throughput (tok/s)'] / baseline_throughput
    latency_increase = row['Latency/Request (s)'] / baseline_latency
    print(f"Batch {row['Batch Size']:2d}: {throughput_gain:.2f}x throughput, {latency_increase:.2f}x latency")

In [ ]:
# Visualize trade-offs
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Throughput vs Batch Size
axes[0].plot(df_results['Batch Size'], df_results['Throughput (tok/s)'], 
            marker='o', linewidth=2, markersize=8, color='green')
axes[0].set_xlabel('Batch Size', fontsize=12)
axes[0].set_ylabel('Throughput (tokens/sec)', fontsize=12)
axes[0].set_title('Throughput vs Batch Size', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log', base=2)

# Annotate values
for _, row in df_results.iterrows():
    axes[0].annotate(f"{row['Throughput (tok/s)']:.0f}",
                    (row['Batch Size'], row['Throughput (tok/s)']),
                    textcoords="offset points", xytext=(0,10), ha='center')

# Plot 2: Latency vs Batch Size
axes[1].plot(df_results['Batch Size'], df_results['Latency/Request (s)'] * 1000, 
            marker='s', linewidth=2, markersize=8, color='coral')
axes[1].set_xlabel('Batch Size', fontsize=12)
axes[1].set_ylabel('Latency per Request (ms)', fontsize=12)
axes[1].set_title('Latency vs Batch Size', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log', base=2)

# Annotate values
for _, row in df_results.iterrows():
    axes[1].annotate(f"{row['Latency/Request (s)']*1000:.0f}ms",
                    (row['Batch Size'], row['Latency/Request (s)'] * 1000),
                    textcoords="offset points", xytext=(0,10), ha='center')

# Plot 3: Efficiency (throughput/latency trade-off)
throughput_normalized = df_results['Throughput (tok/s)'] / df_results['Throughput (tok/s)'].iloc[0]
latency_normalized = df_results['Latency/Request (s)'] / df_results['Latency/Request (s)'].iloc[0]

axes[2].plot(df_results['Batch Size'], throughput_normalized, 
            marker='o', linewidth=2, markersize=8, label='Throughput gain', color='green')
axes[2].plot(df_results['Batch Size'], latency_normalized, 
            marker='s', linewidth=2, markersize=8, label='Latency increase', color='coral')
axes[2].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Batch Size', fontsize=12)
axes[2].set_ylabel('Relative to Batch Size=1', fontsize=12)
axes[2].set_title('Throughput Gain vs Latency Cost', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_xscale('log', base=2)

plt.tight_layout()
plt.savefig('batching_tradeoffs.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Observations:")
print("1. Throughput increases with batch size (more parallel work)")
print("2. Latency per request also increases (waiting for batch to complete)")
print("3. Diminishing returns: larger batches give less incremental benefit")
print("4. Optimal batch size depends on latency requirements and GPU memory")

## 4. Padding Efficiency Analysis

In [ ]:
def calculate_padding_efficiency(prompts: List[str], tokenizer) -> Dict:
    """Calculate padding efficiency for a batch of prompts."""
    
    # Tokenize each prompt
    token_lengths = []
    for prompt in prompts:
        tokens = tokenizer(prompt, return_tensors="pt").input_ids[0]
        token_lengths.append(len(tokens))
    
    max_length = max(token_lengths)
    total_actual_tokens = sum(token_lengths)
    total_with_padding = max_length * len(prompts)
    padding_tokens = total_with_padding - total_actual_tokens
    efficiency = (total_actual_tokens / total_with_padding) * 100
    
    return {
        'num_prompts': len(prompts),
        'max_length': max_length,
        'total_actual_tokens': total_actual_tokens,
        'total_with_padding': total_with_padding,
        'padding_tokens': padding_tokens,
        'efficiency_pct': efficiency,
        'token_lengths': token_lengths,
    }


def compare_batching_strategies(prompts: List[str], tokenizer, batch_sizes: List[int]):
    """Compare padding efficiency for different batching strategies."""
    
    results = []
    
    for batch_size in batch_sizes:
        num_batches = len(prompts) // batch_size
        
        if num_batches == 0:
            continue
        
        total_efficiency = []
        total_padding = 0
        total_actual = 0
        
        for i in range(num_batches):
            batch = prompts[i * batch_size : (i + 1) * batch_size]
            metrics = calculate_padding_efficiency(batch, tokenizer)
            total_efficiency.append(metrics['efficiency_pct'])
            total_padding += metrics['padding_tokens']
            total_actual += metrics['total_actual_tokens']
        
        avg_efficiency = np.mean(total_efficiency)
        
        results.append({
            'Batch Size': batch_size,
            'Num Batches': num_batches,
            'Avg Efficiency (%)': avg_efficiency,
            'Total Padding Tokens': total_padding,
            'Total Actual Tokens': total_actual,
            'Padding Overhead (%)': (total_padding / (total_padding + total_actual)) * 100,
        })
    
    return pd.DataFrame(results)


# Create prompts with varying lengths
varying_prompts = [
    "AI",
    "The quick brown fox",
    "Machine learning is revolutionizing technology",
    "In the field of artificial intelligence, researchers are constantly pushing boundaries",
    "Hello world",
    "Deep learning models",
    "Natural language processing enables computers to understand",
    "The future of autonomous vehicles depends on advances in computer vision and sensor fusion technology",
] * 4  # Repeat to get more samples

print(f"Analyzing padding efficiency with {len(varying_prompts)} prompts...\n")
print("="*80)

batch_sizes = [1, 2, 4, 8]
df_padding = compare_batching_strategies(varying_prompts, tokenizer, batch_sizes)

print("\nPadding Efficiency by Batch Size")
print("="*80)
print(df_padding.to_string(index=False))

print("\n" + "="*80)
print("\nInsights:")
print(f"  - Larger batches = more padding overhead (lower efficiency)")
print(f"  - Best efficiency: batch_size={df_padding.loc[df_padding['Avg Efficiency (%)'].idxmax(), 'Batch Size']:.0f} ({df_padding['Avg Efficiency (%)'].max():.1f}%)")
print(f"  - Worst efficiency: batch_size={df_padding.loc[df_padding['Avg Efficiency (%)'].idxmin(), 'Batch Size']:.0f} ({df_padding['Avg Efficiency (%)'].min():.1f}%)")

In [ ]:
# Visualize padding efficiency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Efficiency vs Batch Size
axes[0].plot(df_padding['Batch Size'], df_padding['Avg Efficiency (%)'], 
            marker='o', linewidth=2, markersize=8, color='purple')
axes[0].set_xlabel('Batch Size', fontsize=12)
axes[0].set_ylabel('Average Efficiency (%)', fontsize=12)
axes[0].set_title('Padding Efficiency vs Batch Size', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=80, color='green', linestyle='--', alpha=0.5, label='80% threshold')
axes[0].legend()

# Annotate values
for _, row in df_padding.iterrows():
    axes[0].annotate(f"{row['Avg Efficiency (%)']:.1f}%",
                    (row['Batch Size'], row['Avg Efficiency (%)']),
                    textcoords="offset points", xytext=(0,10), ha='center')

# Plot 2: Padding Overhead
x = np.arange(len(df_padding))
width = 0.35

axes[1].bar(x, df_padding['Total Actual Tokens'], width, label='Actual tokens', color='lightblue')
axes[1].bar(x, df_padding['Total Padding Tokens'], width, bottom=df_padding['Total Actual Tokens'], 
           label='Padding tokens', color='coral', alpha=0.6)

axes[1].set_xlabel('Batch Size', fontsize=12)
axes[1].set_ylabel('Total Tokens', fontsize=12)
axes[1].set_title('Actual vs Padding Tokens', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_padding['Batch Size'])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('padding_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Finding:")
print("Padding efficiency decreases with larger batches due to length variance!")

## 5. Finding Optimal Batch Size

In [ ]:
def find_optimal_batch_size(
    model,
    tokenizer,
    sample_prompts: List[str],
    max_latency_ms: float = 1000.0,
    max_new_tokens: int = 50,
) -> Dict:
    """
    Find optimal batch size given latency constraint.
    
    Args:
        max_latency_ms: Maximum acceptable latency per request in milliseconds
    
    Returns:
        Dict with optimal configuration
    """
    
    batch_sizes = [1, 2, 4, 8, 16, 32]
    results = []
    
    print(f"Finding optimal batch size (max latency: {max_latency_ms}ms)...\n")
    
    for batch_size in batch_sizes:
        # Test if we have enough prompts
        if len(sample_prompts) < batch_size:
            break
        
        batch = sample_prompts[:batch_size]
        
        # Measure latency
        start = time.time()
        _ = generate_batched(model, tokenizer, batch, max_new_tokens)
        if device == "cuda":
            torch.cuda.synchronize()
        total_time = time.time() - start
        
        latency_per_request_ms = (total_time / batch_size) * 1000
        throughput = (batch_size * max_new_tokens) / total_time
        
        meets_latency = latency_per_request_ms <= max_latency_ms
        
        results.append({
            'batch_size': batch_size,
            'latency_ms': latency_per_request_ms,
            'throughput': throughput,
            'meets_constraint': meets_latency,
        })
        
        status = "✓" if meets_latency else "✗"
        print(f"Batch {batch_size:2d}: {latency_per_request_ms:6.1f}ms latency, "
              f"{throughput:6.1f} tok/s throughput {status}")
        
        # Stop if we exceed latency constraint
        if not meets_latency:
            break
    
    # Find optimal (highest throughput meeting constraint)
    valid_results = [r for r in results if r['meets_constraint']]
    
    if not valid_results:
        return {
            'optimal_batch_size': 1,
            'latency_ms': results[0]['latency_ms'],
            'throughput': results[0]['throughput'],
            'note': 'Even batch_size=1 exceeds latency constraint'
        }
    
    optimal = max(valid_results, key=lambda x: x['throughput'])
    
    return {
        'optimal_batch_size': optimal['batch_size'],
        'latency_ms': optimal['latency_ms'],
        'throughput': optimal['throughput'],
        'all_results': results,
    }


# Find optimal for different latency requirements
latency_requirements = [500, 1000, 2000]  # milliseconds

test_prompts_opt = [
    "The future of artificial intelligence",
    "Machine learning enables us to",
    "Deep neural networks are",
    "Natural language processing helps",
] * 10

print("\n" + "="*80)
print("Optimal Batch Size for Different Latency Requirements")
print("="*80)

optimal_configs = []

for max_latency in latency_requirements:
    print(f"\n\nLatency requirement: {max_latency}ms")
    print("-" * 80)
    
    config = find_optimal_batch_size(
        model, tokenizer, test_prompts_opt,
        max_latency_ms=max_latency,
        max_new_tokens=50
    )
    
    optimal_configs.append({
        'Max Latency (ms)': max_latency,
        'Optimal Batch Size': config['optimal_batch_size'],
        'Actual Latency (ms)': config['latency_ms'],
        'Throughput (tok/s)': config['throughput'],
    })
    
    print(f"\n→ Optimal batch size: {config['optimal_batch_size']}")
    print(f"  Latency: {config['latency_ms']:.1f}ms")
    print(f"  Throughput: {config['throughput']:.1f} tokens/sec")

df_optimal = pd.DataFrame(optimal_configs)
print("\n\n" + "="*80)
print("Summary")
print("="*80)
print(df_optimal.to_string(index=False))

## 6. Production Best Practices

In [ ]:
# Production batching implementation
production_code = '''
# Production static batching with best practices

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List
import time

class BatchedInferenceEngine:
    """Production-ready batched inference engine."""
    
    def __init__(
        self,
        model_name: str,
        batch_size: int = 8,
        max_new_tokens: int = 50,
        device: str = "cuda",
    ):
        self.batch_size = batch_size
        self.max_new_tokens = max_new_tokens
        self.device = device
        
        # Load model with optimal precision
        dtype = torch.float16 if device == "cuda" else torch.float32
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map="auto" if device == "cuda" else None,
        )
        self.model.eval()
        
        if device == "cpu":
            self.model = self.model.to(device)
    
    def generate_batch(self, prompts: List[str]) -> List[str]:
        """Generate for a single batch."""
        
        inputs = self.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,  # Prevent excessive padding
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                use_cache=True,  # Always use KV cache
                pad_token_id=self.tokenizer.eos_token_id,
                do_sample=False,
            )
        
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
    def generate(self, prompts: List[str]) -> List[str]:
        """Generate for multiple prompts with automatic batching."""
        
        all_outputs = []
        
        # Process in batches
        for i in range(0, len(prompts), self.batch_size):
            batch = prompts[i:i + self.batch_size]
            batch_outputs = self.generate_batch(batch)
            all_outputs.extend(batch_outputs)
        
        return all_outputs


# Usage example
if __name__ == "__main__":
    # Initialize engine with optimal batch size
    engine = BatchedInferenceEngine(
        model_name="gpt2",
        batch_size=8,  # Determined from latency analysis
        max_new_tokens=50,
    )
    
    # Generate for multiple prompts
    prompts = [
        "The future of AI is",
        "Machine learning enables",
        "Deep learning can",
        # ... more prompts
    ]
    
    start = time.time()
    outputs = engine.generate(prompts)
    elapsed = time.time() - start
    
    print(f"Generated {len(outputs)} responses in {elapsed:.2f}s")
    print(f"Throughput: {len(outputs)/elapsed:.2f} requests/sec")
'''

print("Production Batching Implementation")
print("="*100)
print(production_code)
print("="*100)

print("\n\nProduction Checklist:")
checklist = [
    "✓ Determine optimal batch size based on latency requirements",
    "✓ Set max_length to prevent excessive padding",
    "✓ Always enable use_cache=True for KV caching",
    "✓ Use FP16/BF16 for memory efficiency",
    "✓ Monitor padding efficiency in production",
    "✓ Consider sorting prompts by length to minimize padding",
    "✓ Implement request queuing for automatic batching",
    "✓ Set timeout for batch accumulation",
    "✓ Track throughput and latency metrics",
    "✓ Auto-scale batch size based on GPU memory",
]

for item in checklist:
    print(f"  {item}")

## Summary and Key Takeaways

### Performance Results
- **Throughput**: 2-8x improvement depending on batch size
- **Latency trade-off**: Increases proportionally with batch size
- **Padding overhead**: 10-50% depending on prompt length variance

### Core Concepts
1. **Static batching**: Process multiple requests together
2. **Padding**: Required to make all sequences same length
3. **Efficiency**: Actual tokens / Total tokens (including padding)
4. **Trade-off**: Higher throughput vs higher latency

### Optimal Batch Size Selection
- **For real-time apps**: batch_size=1-4 (low latency priority)
- **For high throughput**: batch_size=8-32 (throughput priority)
- **For offline batch**: batch_size=32+ (maximum throughput)
- **Constrain by**: GPU memory, latency requirements, padding efficiency

### Best Practices
1. **Profile first**: Measure latency and throughput for your workload
2. **Set constraints**: Define maximum acceptable latency
3. **Optimize padding**: Sort prompts by length when possible
4. **Monitor efficiency**: Track padding overhead in production
5. **Dynamic batching**: Consider continuous batching (Stage 2)

### Limitations of Static Batching
1. **Padding waste**: Can be 20-50% with varying lengths
2. **Latency**: All requests wait for slowest in batch
3. **Memory**: Padding increases KV cache size
4. **Inflexibility**: Fixed batch size regardless of load

### When Static Batching Works Best
- Similar length prompts (high padding efficiency)
- Offline/batch processing workloads
- Throughput more important than latency
- Predictable, steady request rate

### Next Steps
- **Stage 1**: torch.compile (Notebook 13) - 1.2-1.5x additional speedup
- **Stage 1**: INT8 Quantization (Notebook 14) - Memory reduction
- **Stage 2**: Continuous Batching - Better than static batching
- **Stage 2**: PagedAttention - More efficient memory management

### References
- LLM_INFERENCE_ROADMAP.md - Stage 1: Basic Optimizations
- Orca: https://www.usenix.org/conference/osdi22/presentation/yu (continuous batching)
- vLLM: https://arxiv.org/abs/2309.06180 (PagedAttention)